# GNN (Feeder) + ResNet1D-LSTM — Late Fusion Tamper Detection

Three independently-trained pieces, fused only at the very end:

1. **CNN-LSTM branch** (unchanged from before): ResNet1D extractor → LSTM classifier,
   trained on each meter's own continuous history. Not affected by cross-meter data
   quality — a meter's own windows only depend on its own past readings.
2. **GNN branch** (new): every meter connects to one shared virtual feeder node — a star
   graph, since all 43 meters sit on the same feeder. A single message-passing layer is
   enough (meter → feeder → meter is the whole graph in 2 hops; there's no deeper
   structure to learn). Produces a per-meter, per-timestamp embedding reflecting "what did
   the shared feeder look like right now."
3. **Late fusion**: both branches are frozen after their own training, embeddings are
   extracted via forward pass only, concatenated, and a small final classifier is trained
   on the combined features.

## Data-quality handling (per your direction)
- `smart_meter_labeled.csv` is **read-only input** — never rewritten. Severe-dropout
  timestamps stay in it.
- Severe-dropout timestamps (too few meters reporting to represent the feeder's actual
  state) are excluded from the **running/in-memory GNN training data** via a boolean
  mask — not deleted from disk.
- The CNN-LSTM branch is unaffected either way — it never depends on how many other
  meters reported.

## Required correction for the fusion to be well-defined
Fusing two independently-trained branches per (meter, timestamp) sample only makes sense
if both branches agree on which samples are "train" and which are "test." So both
branches now split on **one shared global cutoff timestamp**, instead of each computing
its own fractional split independently (which is what the separate notebooks did before,
and would leave fusion samples inconsistently labeled train vs. test across branches).


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

Using device: mps


## Config

In [8]:
TIME_COL = "Load Survey Time"
ID_COL   = "MID"

CURRENT_COLS = ["R Phase Current", "Y Phase Current", "B Phase Current"]
VOLTAGE_COLS = ["R Phase Voltage", "Y Phase Voltage", "B Phase Voltage"]
ENERGY_COLS  = ["Active Energy", "Apparent Energy", "Reactive Lag", "Reactive Lead"]
PF_COL = "PF"
RAW_FEATURE_COLS = CURRENT_COLS + VOLTAGE_COLS + ENERGY_COLS + [PF_COL]

TARGET_COLS = [
    "Current Imbalance", "Voltage Unbalance", "Missing Potential", "High Voltage",
    "Low Voltage", "Over Current", "Very Low PF", "Neutral Disturbance",
    "CT Reversal", "CT Bypass", "CT Open", "Missing Value",
]

WINDOW_SIZE = 48                 # CNN-LSTM branch — past readings per window
RESNET_FILTERS = [32, 64, 64]
LSTM_UNITS = 64

# GNN branch
MIN_METERS_FOR_GNN = 30          # timestamps with fewer reporting meters than this are
                                  # "severe dropout" — excluded from GNN/fusion training,
                                  # kept untouched in the saved CSV. Based on the observed
                                  # distribution (a cliff from 42 down to 33, then a long
                                  # thin tail) — adjust if your data's cliff sits elsewhere.
GNN_HIDDEN_DIM = 32

BATCH_SIZE = 256
EPOCHS = 15
LEARNING_RATE = 1e-3
GLOBAL_TRAIN_FRAC = 0.8          # single shared cutoff used by BOTH branches

## Load labeled dataset
Read-only — never rewritten.

In [9]:
df = pd.read_csv("Dataset/clean_data_labeled.csv")
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
df = df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)
df[RAW_FEATURE_COLS] = df[RAW_FEATURE_COLS].fillna(0)

all_meters = sorted(df[ID_COL].unique())
n_meters = len(all_meters)
meter_to_idx = {m: i for i, m in enumerate(all_meters)}
print(f"Meters: {n_meters}")

# --- Single shared global cutoff, used by BOTH branches ---
GLOBAL_CUTOFF_TIME = df[TIME_COL].quantile(GLOBAL_TRAIN_FRAC)
print(f"Global train/test cutoff: {GLOBAL_CUTOFF_TIME}")

Meters: 43
Global train/test cutoff: 2017-11-29 03:00:00


## Step 1 — Rolling-window tabular features (CNN-LSTM branch)

In [10]:
def add_rolling_features(df: pd.DataFrame, window: int = 8) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby(ID_COL)[RAW_FEATURE_COLS]
    roll_mean = grouped.transform(lambda s: s.rolling(window, min_periods=1).mean())
    roll_std  = grouped.transform(lambda s: s.rolling(window, min_periods=1).std().fillna(0))
    roll_mean.columns = [f"{c}_roll_mean" for c in RAW_FEATURE_COLS]
    roll_std.columns  = [f"{c}_roll_std"  for c in RAW_FEATURE_COLS]
    return pd.concat([df, roll_mean, roll_std], axis=1)

df = add_rolling_features(df)
ROLLING_COLS = [c for c in df.columns if c.endswith("_roll_mean") or c.endswith("_roll_std")]
TABULAR_COLS = RAW_FEATURE_COLS + ROLLING_COLS

## Step 2 — CNN-LSTM branch: causal windows per meter
Built from each meter's own FULL continuous history — unaffected by cross-meter dropout.
Train/test split now uses the shared `GLOBAL_CUTOFF_TIME`, not a per-meter fraction.

In [11]:
def build_windows(df: pd.DataFrame):
    X_seq, X_tab, Y, meta = [], [], [], []

    for mid, g in df.groupby(ID_COL):
        g = g.sort_values(TIME_COL).reset_index(drop=True)
        raw_vals = g[RAW_FEATURE_COLS].values
        tab_vals = g[TABULAR_COLS].values
        y_vals   = g[TARGET_COLS].values
        times    = g[TIME_COL].values

        for i in range(WINDOW_SIZE - 1, len(g)):
            X_seq.append(raw_vals[i - WINDOW_SIZE + 1: i + 1])
            X_tab.append(tab_vals[i])
            Y.append(y_vals[i])
            meta.append((mid, times[i]))

    return (np.array(X_seq, dtype=np.float32),
            np.array(X_tab, dtype=np.float32),
            np.array(Y, dtype=np.float32),
            pd.DataFrame(meta, columns=[ID_COL, TIME_COL]))

X_seq, X_tab, Y, meta = build_windows(df)
print(f"CNN-LSTM windows: {X_seq.shape[0]} | Sequence shape: {X_seq.shape[1:]} | Tabular: {X_tab.shape[1]} | Targets: {Y.shape[1]}")

meta[TIME_COL] = pd.to_datetime(meta[TIME_COL])
train_mask_cl = (meta[TIME_COL] <= GLOBAL_CUTOFF_TIME).values
test_mask_cl  = ~train_mask_cl

X_seq_train, X_seq_test = X_seq[train_mask_cl], X_seq[test_mask_cl]
X_tab_train, X_tab_test = X_tab[train_mask_cl], X_tab[test_mask_cl]
Y_train, Y_test = Y[train_mask_cl], Y[test_mask_cl]
meta_train, meta_test = meta[train_mask_cl].reset_index(drop=True), meta[test_mask_cl].reset_index(drop=True)

print(f"CNN-LSTM train windows: {len(X_seq_train)} | test windows: {len(X_seq_test)}")

CNN-LSTM windows: 1705270 | Sequence shape: (48, 11) | Tabular: 33 | Targets: 12
CNN-LSTM train windows: 1363850 | test windows: 341420


In [12]:
seq_scaler = StandardScaler()
n_train, w, c = X_seq_train.shape
seq_scaler.fit(X_seq_train.reshape(-1, c))
X_seq_train_scaled = seq_scaler.transform(X_seq_train.reshape(-1, c)).reshape(n_train, w, c)
X_seq_test_scaled  = seq_scaler.transform(X_seq_test.reshape(-1, c)).reshape(X_seq_test.shape[0], w, c)

tab_scaler = StandardScaler()
X_tab_train_scaled = tab_scaler.fit_transform(X_tab_train)
X_tab_test_scaled  = tab_scaler.transform(X_tab_test)

val_frac = 0.15
val_split = int(len(X_seq_train_scaled) * (1 - val_frac))
X_seq_fit, X_seq_val = X_seq_train_scaled[:val_split], X_seq_train_scaled[val_split:]
X_tab_fit, X_tab_val = X_tab_train_scaled[:val_split], X_tab_train_scaled[val_split:]
Y_fit, Y_val = Y_train[:val_split], Y_train[val_split:]

## Step 3 — ResNet1D + LSTM (CNN-LSTM branch), same architecture as before
`padding="same"`, stride 1 throughout preserves the window length so the LSTM gets a real
sequence, not a pooled vector. The `LSTMClassifier` now also returns its pre-output fused
representation (`fused_embedding`) — this is the piece the late-fusion stage will reuse.

In [13]:
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.shortcut = None
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x):
        residual = x if self.shortcut is None else self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + residual)


class ResNet1DExtractor(nn.Module):
    def __init__(self, in_channels, filters):
        super().__init__()
        blocks, prev = [], in_channels
        for f in filters:
            blocks.append(ResidualBlock1D(prev, f))
            prev = f
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.blocks(x)
        return x.transpose(1, 2)


class ResNet1DStage1(nn.Module):
    def __init__(self, in_channels, filters, n_targets):
        super().__init__()
        self.extractor = ResNet1DExtractor(in_channels, filters)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(filters[-1], n_targets)

    def forward(self, x):
        embed_seq = self.extractor(x)
        pooled = self.pool(embed_seq.transpose(1, 2)).squeeze(-1)
        return self.head(pooled)


class LSTMClassifier(nn.Module):
    def __init__(self, embed_dim, n_tabular, lstm_units, n_targets):
        super().__init__()
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=lstm_units, batch_first=True)
        self.fc1 = nn.Linear(lstm_units + n_tabular, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, n_targets)

    def forward(self, embed_seq, tab_features):
        _, (h_n, _) = self.lstm(embed_seq)
        lstm_out = h_n.squeeze(0)
        combined = torch.cat([lstm_out, tab_features], dim=1)
        fused_embedding = self.relu(self.fc1(combined))   # <-- reused by the fusion stage
        logits = self.fc2(fused_embedding)
        return logits, fused_embedding

In [14]:
class WindowDataset(Dataset):
    def __init__(self, X_seq, Y):
        self.X_seq = torch.tensor(X_seq, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X_seq)
    def __getitem__(self, idx): return self.X_seq[idx], self.Y[idx]


n_features = X_seq.shape[2]
n_targets = Y.shape[1]

train_ds = WindowDataset(X_seq_fit, Y_fit)
val_ds   = WindowDataset(X_seq_val, Y_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

resnet_stage1 = ResNet1DStage1(n_features, RESNET_FILTERS, n_targets).to(device)
optimizer = torch.optim.Adam(resnet_stage1.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(EPOCHS):
    resnet_stage1.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(resnet_stage1(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_ds)

    resnet_stage1.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += criterion(resnet_stage1(xb), yb).item() * xb.size(0)
    val_loss /= len(val_ds)
    print(f"[ResNet1D] Epoch {epoch+1}/{EPOCHS} — train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}")

[ResNet1D] Epoch 1/15 — train_loss: 0.0373  val_loss: 0.0336
[ResNet1D] Epoch 2/15 — train_loss: 0.0192  val_loss: 0.0258
[ResNet1D] Epoch 3/15 — train_loss: 0.0150  val_loss: 0.0220
[ResNet1D] Epoch 4/15 — train_loss: 0.0125  val_loss: 0.0239
[ResNet1D] Epoch 5/15 — train_loss: 0.0109  val_loss: 0.0189
[ResNet1D] Epoch 6/15 — train_loss: 0.0098  val_loss: 0.0216
[ResNet1D] Epoch 7/15 — train_loss: 0.0090  val_loss: 0.0205
[ResNet1D] Epoch 8/15 — train_loss: 0.0083  val_loss: 0.0172
[ResNet1D] Epoch 9/15 — train_loss: 0.0078  val_loss: 0.0140
[ResNet1D] Epoch 10/15 — train_loss: 0.0073  val_loss: 0.0129
[ResNet1D] Epoch 11/15 — train_loss: 0.0070  val_loss: 0.0140
[ResNet1D] Epoch 12/15 — train_loss: 0.0067  val_loss: 0.0155
[ResNet1D] Epoch 13/15 — train_loss: 0.0064  val_loss: 0.0147
[ResNet1D] Epoch 14/15 — train_loss: 0.0061  val_loss: 0.0231
[ResNet1D] Epoch 15/15 — train_loss: 0.0060  val_loss: 0.0140


In [15]:
resnet_stage1.eval()
extractor = resnet_stage1.extractor  # frozen from here on

def extract_embeddings(X_seq_scaled, batch_size=BATCH_SIZE):
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(X_seq_scaled), batch_size):
            batch = torch.tensor(X_seq_scaled[i:i+batch_size], dtype=torch.float32).to(device)
            embeddings.append(extractor(batch).cpu().numpy())
    return np.concatenate(embeddings, axis=0)

train_embed_seq = extract_embeddings(X_seq_train_scaled)
test_embed_seq  = extract_embeddings(X_seq_test_scaled)
print(f"Embedding sequences — train: {train_embed_seq.shape}, test: {test_embed_seq.shape}")

Embedding sequences — train: (1363850, 48, 64), test: (341420, 48, 64)


In [16]:
class EmbeddingTabDataset(Dataset):
    def __init__(self, embed_seq, tab, Y):
        self.embed_seq = torch.tensor(embed_seq, dtype=torch.float32)
        self.tab = torch.tensor(tab, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.embed_seq)
    def __getitem__(self, idx): return self.embed_seq[idx], self.tab[idx], self.Y[idx]


embed_dim = train_embed_seq.shape[-1]
n_tabular = X_tab_train_scaled.shape[1]
train_embed_fit, train_embed_val = train_embed_seq[:val_split], train_embed_seq[val_split:]

final_train_ds = EmbeddingTabDataset(train_embed_fit, X_tab_fit, Y_fit)
final_val_ds   = EmbeddingTabDataset(train_embed_val, X_tab_val, Y_val)
final_train_loader = DataLoader(final_train_ds, batch_size=BATCH_SIZE, shuffle=True)
final_val_loader   = DataLoader(final_val_ds, batch_size=BATCH_SIZE, shuffle=False)

lstm_classifier = LSTMClassifier(embed_dim, n_tabular, LSTM_UNITS, n_targets).to(device)
final_optimizer = torch.optim.Adam(lstm_classifier.parameters(), lr=LEARNING_RATE)
final_criterion = nn.BCEWithLogitsLoss()

for epoch in range(EPOCHS):
    lstm_classifier.train()
    train_loss = 0.0
    for embed_b, tab_b, yb in final_train_loader:
        embed_b, tab_b, yb = embed_b.to(device), tab_b.to(device), yb.to(device)
        final_optimizer.zero_grad()
        logits, _ = lstm_classifier(embed_b, tab_b)
        loss = final_criterion(logits, yb)
        loss.backward()
        final_optimizer.step()
        train_loss += loss.item() * embed_b.size(0)
    train_loss /= len(final_train_ds)

    lstm_classifier.eval()
    val_loss = 0.0
    with torch.no_grad():
        for embed_b, tab_b, yb in final_val_loader:
            embed_b, tab_b, yb = embed_b.to(device), tab_b.to(device), yb.to(device)
            logits, _ = lstm_classifier(embed_b, tab_b)
            val_loss += final_criterion(logits, yb).item() * embed_b.size(0)
    val_loss /= len(final_val_ds)
    print(f"[LSTM] Epoch {epoch+1}/{EPOCHS} — train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}")

[LSTM] Epoch 1/15 — train_loss: 0.0101  val_loss: 0.0095
[LSTM] Epoch 2/15 — train_loss: 0.0043  val_loss: 0.0090
[LSTM] Epoch 3/15 — train_loss: 0.0039  val_loss: 0.0090
[LSTM] Epoch 4/15 — train_loss: 0.0037  val_loss: 0.0092
[LSTM] Epoch 5/15 — train_loss: 0.0036  val_loss: 0.0088
[LSTM] Epoch 6/15 — train_loss: 0.0035  val_loss: 0.0086
[LSTM] Epoch 7/15 — train_loss: 0.0034  val_loss: 0.0096
[LSTM] Epoch 8/15 — train_loss: 0.0034  val_loss: 0.0092
[LSTM] Epoch 9/15 — train_loss: 0.0033  val_loss: 0.0094
[LSTM] Epoch 10/15 — train_loss: 0.0032  val_loss: 0.0088
[LSTM] Epoch 11/15 — train_loss: 0.0032  val_loss: 0.0091
[LSTM] Epoch 12/15 — train_loss: 0.0031  val_loss: 0.0096
[LSTM] Epoch 13/15 — train_loss: 0.0031  val_loss: 0.0092
[LSTM] Epoch 14/15 — train_loss: 0.0030  val_loss: 0.0097
[LSTM] Epoch 15/15 — train_loss: 0.0030  val_loss: 0.0102


## Step 4 — Severe-dropout diagnostic (running dataset only, CSV untouched)
Identifies which global timestamps have too few reporting meters to represent the
feeder's actual state. These get excluded from GNN/fusion training in memory — the
underlying CSV is never modified.

In [17]:
meters_per_timestamp = df.groupby(TIME_COL)[ID_COL].nunique()
usable_timestamps = meters_per_timestamp[meters_per_timestamp >= MIN_METERS_FOR_GNN].index

print(f"Total unique timestamps: {len(meters_per_timestamp)}")
print(f"Usable for GNN (>= {MIN_METERS_FOR_GNN} meters reporting): {len(usable_timestamps)}")
print(f"Excluded as severe dropout (kept in CSV, dropped from GNN training only): "
      f"{len(meters_per_timestamp) - len(usable_timestamps)}")

Total unique timestamps: 39860
Usable for GNN (>= 30 meters reporting): 39845
Excluded as severe dropout (kept in CSV, dropped from GNN training only): 15


## Step 5 — Build per-timestamp graph snapshots
For every usable timestamp: a feature vector per meter (forward-filled from that meter's
own last known reading if it didn't report exactly at this timestamp, plus a presence
flag so the GNN knows which values are stale) and one aggregated feeder feature vector.

In [18]:
pivot = df.pivot_table(index=TIME_COL, columns=ID_COL, values=RAW_FEATURE_COLS)
presence = (df.assign(_present=1)
              .pivot_table(index=TIME_COL, columns=ID_COL, values="_present", fill_value=0))

# Forward-fill each meter's own readings across the shared time grid, then zero-fill any
# still-missing leading values (meter had no prior reading yet)
pivot = pivot.sort_index().ffill().fillna(0)
presence = presence.reindex(pivot.index).fillna(0)

pivot = pivot.loc[usable_timestamps]
presence = presence.loc[usable_timestamps]

n_feat = len(RAW_FEATURE_COLS)
node_feat_dim = n_feat + 1  # +1 for the presence flag

meter_features = np.zeros((len(usable_timestamps), n_meters, node_feat_dim), dtype=np.float32)
for m in all_meters:
    idx = meter_to_idx[m]
    meter_features[:, idx, :n_feat] = pivot[[(f, m) for f in RAW_FEATURE_COLS]].values
    meter_features[:, idx, n_feat] = presence[m].values

# Feeder features: mean of PRESENT meters at each timestamp, + fraction present
present_counts = presence.values.sum(axis=1, keepdims=True)
present_counts_safe = np.clip(present_counts, 1, None)
feeder_raw = (meter_features[:, :, :n_feat] * presence.values[:, :, None]).sum(axis=1) / present_counts_safe
feeder_frac_present = present_counts / n_meters
feeder_features = np.concatenate([feeder_raw, feeder_frac_present], axis=1).astype(np.float32)

print(f"meter_features: {meter_features.shape} | feeder_features: {feeder_features.shape}")

meter_features: (39845, 43, 12) | feeder_features: (39845, 12)


## Step 6 — GNN labels
Per (meter, timestamp) — only for meters that ACTUALLY reported at that timestamp (the
presence mask also controls which samples get a supervised label; forward-filled stand-in
values are used for message passing context only, never as a labeled training sample).

In [19]:
label_lookup = df.set_index([TIME_COL, ID_COL])[TARGET_COLS]

gnn_labels = np.zeros((len(usable_timestamps), n_meters, len(TARGET_COLS)), dtype=np.float32)
for t_idx, t in enumerate(usable_timestamps):
    for m in all_meters:
        if presence.loc[t, m] == 1:
            gnn_labels[t_idx, meter_to_idx[m]] = label_lookup.loc[(t, m)].values

usable_timestamps_dt = pd.to_datetime(usable_timestamps)
gnn_train_mask_t = (usable_timestamps_dt <= GLOBAL_CUTOFF_TIME)
gnn_test_mask_t  = ~gnn_train_mask_t

print(f"GNN train timestamps: {gnn_train_mask_t.sum()} | test timestamps: {gnn_test_mask_t.sum()}")

GNN train timestamps: 31885 | test timestamps: 7960


## Step 7 — Feeder GNN module (star graph, single hop)
One message-passing layer: meters attend into the feeder (weighted by an attention score,
masked so absent meters don't contribute), the feeder's updated representation is
broadcast back to every meter. One layer is enough — meter→feeder→meter is the entire
graph in 2 hops; there's no deeper structure for more layers to find here.

In [20]:
class FeederGATLayer(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.meter_proj = nn.Linear(in_dim, hidden_dim)
        self.feeder_proj = nn.Linear(in_dim, hidden_dim)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.update_meter = nn.Linear(hidden_dim * 2, hidden_dim)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, meter_feats, feeder_feats, presence_mask):
        # meter_feats: (batch, n_meters, in_dim) | feeder_feats: (batch, in_dim)
        # presence_mask: (batch, n_meters), 1 = actually reported at this timestamp
        batch, n_m, _ = meter_feats.shape
        h_meter = self.meter_proj(meter_feats)                       # (batch, n_meters, hidden)
        h_feeder = self.feeder_proj(feeder_feats)                    # (batch, hidden)
        h_feeder_exp = h_feeder.unsqueeze(1).expand(-1, n_m, -1)

        # meter -> feeder: attention-weighted aggregation, absent meters masked out
        attn_scores = self.leaky_relu(self.attn(torch.cat([h_meter, h_feeder_exp], dim=-1))).squeeze(-1)
        attn_scores = attn_scores.masked_fill(presence_mask == 0, float("-inf"))
        attn_weights = torch.softmax(attn_scores, dim=1)
        feeder_update = torch.einsum("bn,bnh->bh", attn_weights, h_meter)

        # feeder -> meter: broadcast back, combined with each meter's own projection
        feeder_update_exp = feeder_update.unsqueeze(1).expand(-1, n_m, -1)
        meter_embeddings = torch.relu(
            self.update_meter(torch.cat([h_meter, feeder_update_exp], dim=-1))
        )
        return meter_embeddings, feeder_update


class GNNStage1(nn.Module):
    """GNN layer + a temporary per-meter head, used only to train the GNN embeddings."""
    def __init__(self, in_dim, hidden_dim, n_targets):
        super().__init__()
        self.gnn = FeederGATLayer(in_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, n_targets)

    def forward(self, meter_feats, feeder_feats, presence_mask):
        meter_embeddings, _ = self.gnn(meter_feats, feeder_feats, presence_mask)
        logits = self.head(meter_embeddings)   # (batch, n_meters, n_targets)
        return logits, meter_embeddings

## Step 8 — Train the GNN
Loss is computed **only over meters that actually reported** at each timestamp (masked
out via `presence_mask`) — forward-filled stand-in values still participate in message
passing (so the feeder aggregation is representative even with a meter or two missing),
but they never contribute a fabricated label to the loss.

In [21]:
class GraphSnapshotDataset(Dataset):
    def __init__(self, meter_feats, feeder_feats, presence, labels):
        self.meter_feats = torch.tensor(meter_feats, dtype=torch.float32)
        self.feeder_feats = torch.tensor(feeder_feats, dtype=torch.float32)
        self.presence = torch.tensor(presence, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
    def __len__(self): return len(self.meter_feats)
    def __getitem__(self, idx):
        return self.meter_feats[idx], self.feeder_feats[idx], self.presence[idx], self.labels[idx]


presence_arr = presence.values.astype(np.float32)

gnn_train_ds = GraphSnapshotDataset(
    meter_features[gnn_train_mask_t], feeder_features[gnn_train_mask_t],
    presence_arr[gnn_train_mask_t], gnn_labels[gnn_train_mask_t])
gnn_test_ds = GraphSnapshotDataset(
    meter_features[gnn_test_mask_t], feeder_features[gnn_test_mask_t],
    presence_arr[gnn_test_mask_t], gnn_labels[gnn_test_mask_t])

gnn_train_loader = DataLoader(gnn_train_ds, batch_size=64, shuffle=True)
gnn_test_loader  = DataLoader(gnn_test_ds, batch_size=64, shuffle=False)

gnn_stage1 = GNNStage1(node_feat_dim, GNN_HIDDEN_DIM, n_targets).to(device)
gnn_optimizer = torch.optim.Adam(gnn_stage1.parameters(), lr=LEARNING_RATE)
gnn_criterion = nn.BCEWithLogitsLoss(reduction="none")

def masked_bce_loss(logits, labels, presence_mask):
    # logits/labels: (batch, n_meters, n_targets) | presence_mask: (batch, n_meters)
    per_element_loss = gnn_criterion(logits, labels)               # (batch, n_meters, n_targets)
    mask = presence_mask.unsqueeze(-1).expand_as(per_element_loss)  # broadcast to targets
    masked_loss = per_element_loss * mask
    return masked_loss.sum() / mask.sum().clamp(min=1)

for epoch in range(EPOCHS):
    gnn_stage1.train()
    train_loss = 0.0
    for mf, ff, pr, yb in gnn_train_loader:
        mf, ff, pr, yb = mf.to(device), ff.to(device), pr.to(device), yb.to(device)
        gnn_optimizer.zero_grad()
        logits, _ = gnn_stage1(mf, ff, pr)
        loss = masked_bce_loss(logits, yb, pr)
        loss.backward()
        gnn_optimizer.step()
        train_loss += loss.item() * mf.size(0)
    train_loss /= len(gnn_train_ds)

    gnn_stage1.eval()
    val_loss = 0.0
    with torch.no_grad():
        for mf, ff, pr, yb in gnn_test_loader:
            mf, ff, pr, yb = mf.to(device), ff.to(device), pr.to(device), yb.to(device)
            logits, _ = gnn_stage1(mf, ff, pr)
            val_loss += masked_bce_loss(logits, yb, pr).item() * mf.size(0)
    val_loss /= len(gnn_test_ds)
    print(f"[GNN] Epoch {epoch+1}/{EPOCHS} — train_loss: {train_loss:.4f}  test_loss: {val_loss:.4f}")

[GNN] Epoch 1/15 — train_loss: 0.2852  test_loss: 0.1363
[GNN] Epoch 2/15 — train_loss: 0.1038  test_loss: 0.0798
[GNN] Epoch 3/15 — train_loss: 0.0807  test_loss: 0.0714
[GNN] Epoch 4/15 — train_loss: 0.0742  test_loss: 0.0712
[GNN] Epoch 5/15 — train_loss: 0.0681  test_loss: 0.0606
[GNN] Epoch 6/15 — train_loss: 0.0600  test_loss: 0.0551
[GNN] Epoch 7/15 — train_loss: 0.0547  test_loss: 0.0513
[GNN] Epoch 8/15 — train_loss: 0.0509  test_loss: 0.0489
[GNN] Epoch 9/15 — train_loss: 0.0489  test_loss: 0.0471
[GNN] Epoch 10/15 — train_loss: 0.0470  test_loss: 0.0446
[GNN] Epoch 11/15 — train_loss: 0.0452  test_loss: 0.0451
[GNN] Epoch 12/15 — train_loss: 0.0438  test_loss: 0.0420
[GNN] Epoch 13/15 — train_loss: 0.0425  test_loss: 0.0410
[GNN] Epoch 14/15 — train_loss: 0.0414  test_loss: 0.0415
[GNN] Epoch 15/15 — train_loss: 0.0405  test_loss: 0.0401


## Step 9 — Extract frozen GNN embeddings
Forward pass only, per (meter, timestamp) — same leakage-prevention rule as the CNN-LSTM
branch.

In [22]:
gnn_stage1.eval()
gnn_layer = gnn_stage1.gnn  # frozen from here on

def extract_gnn_embeddings(meter_feats, feeder_feats, presence_mask, batch_size=64):
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(meter_feats), batch_size):
            mf = torch.tensor(meter_feats[i:i+batch_size], dtype=torch.float32).to(device)
            ff = torch.tensor(feeder_feats[i:i+batch_size], dtype=torch.float32).to(device)
            pr = torch.tensor(presence_mask[i:i+batch_size], dtype=torch.float32).to(device)
            emb, _ = gnn_layer(mf, ff, pr)
            embeddings.append(emb.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

gnn_embed_train = extract_gnn_embeddings(
    meter_features[gnn_train_mask_t], feeder_features[gnn_train_mask_t], presence_arr[gnn_train_mask_t])
gnn_embed_test = extract_gnn_embeddings(
    meter_features[gnn_test_mask_t], feeder_features[gnn_test_mask_t], presence_arr[gnn_test_mask_t])

print(f"GNN embeddings — train: {gnn_embed_train.shape}, test: {gnn_embed_test.shape}")
# shape: (n_usable_timestamps, n_meters, GNN_HIDDEN_DIM)

GNN embeddings — train: (31885, 43, 32), test: (7960, 43, 32)


## Step 10 — Build the late-fusion dataset
For each (meter, timestamp) sample that has BOTH a valid CNN-LSTM window AND a GNN
embedding (meter actually present, timestamp not severe-dropout), concatenate the two
branches' embeddings. Samples failing either condition are simply not fusable and are
excluded — consistent with your instruction that severe-dropout timestamps shouldn't
block the CNN-LSTM branch, but also can't be fabricated for the GNN side.

In [23]:
def build_fusion_set(cl_embed, cl_meta, cl_labels, gnn_embed, usable_ts_subset, presence_subset):
    ts_to_pos = {t: i for i, t in enumerate(usable_ts_subset)}
    fused_X, fused_Y = [], []

    for i in range(len(cl_meta)):
        mid, t = cl_meta.iloc[i][ID_COL], cl_meta.iloc[i][TIME_COL]
        if t not in ts_to_pos:
            continue  # timestamp is severe-dropout or otherwise not in the GNN grid
        t_pos = ts_to_pos[t]
        m_idx = meter_to_idx[mid]
        if presence_subset[t_pos, m_idx] == 0:
            continue  # meter didn't actually report at this timestamp — no real GNN label to fuse against

        cl_vec = cl_embed[i]
        gnn_vec = gnn_embed[t_pos, m_idx]
        fused_X.append(np.concatenate([cl_vec, gnn_vec]))
        fused_Y.append(cl_labels[i])

    return np.array(fused_X, dtype=np.float32), np.array(fused_Y, dtype=np.float32)


# CNN-LSTM fused_embeddings (frozen, forward pass only) for every window sample
def extract_cl_embeddings(embed_seq, tab, batch_size=BATCH_SIZE):
    embeddings = []
    lstm_classifier.eval()
    with torch.no_grad():
        for i in range(0, len(embed_seq), batch_size):
            eb = torch.tensor(embed_seq[i:i+batch_size], dtype=torch.float32).to(device)
            tb = torch.tensor(tab[i:i+batch_size], dtype=torch.float32).to(device)
            _, fused = lstm_classifier(eb, tb)
            embeddings.append(fused.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

cl_embed_train_full = extract_cl_embeddings(train_embed_seq, X_tab_train_scaled)
cl_embed_test_full  = extract_cl_embeddings(test_embed_seq, X_tab_test_scaled)

usable_train_ts = usable_timestamps_dt[gnn_train_mask_t]
usable_test_ts  = usable_timestamps_dt[gnn_test_mask_t]
presence_train_ts = presence_arr[gnn_train_mask_t]
presence_test_ts  = presence_arr[gnn_test_mask_t]

X_fusion_train, Y_fusion_train = build_fusion_set(
    cl_embed_train_full, meta_train, Y_train, gnn_embed_train, usable_train_ts, presence_train_ts)
X_fusion_test, Y_fusion_test = build_fusion_set(
    cl_embed_test_full, meta_test, Y_test, gnn_embed_test, usable_test_ts, presence_test_ts)

print(f"Fusion train samples: {X_fusion_train.shape} | Fusion test samples: {X_fusion_test.shape}")

Fusion train samples: (1363850, 96) | Fusion test samples: (341289, 96)


## Step 11 — Final fusion classifier
A small MLP on the concatenated [CNN-LSTM embedding + GNN embedding] — this is the model
that actually produces the final tamper classification.

In [24]:
class FusionClassifier(nn.Module):
    def __init__(self, in_dim, n_targets):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, n_targets)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


fusion_in_dim = X_fusion_train.shape[1]
fusion_model = FusionClassifier(fusion_in_dim, n_targets).to(device)
fusion_optimizer = torch.optim.Adam(fusion_model.parameters(), lr=LEARNING_RATE)
fusion_criterion = nn.BCEWithLogitsLoss()

X_fusion_train_t = torch.tensor(X_fusion_train, dtype=torch.float32)
Y_fusion_train_t = torch.tensor(Y_fusion_train, dtype=torch.float32)
fusion_train_loader = DataLoader(
    list(zip(X_fusion_train_t, Y_fusion_train_t)), batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(EPOCHS):
    fusion_model.train()
    train_loss = 0.0
    for xb, yb in fusion_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        fusion_optimizer.zero_grad()
        loss = fusion_criterion(fusion_model(xb), yb)
        loss.backward()
        fusion_optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(X_fusion_train_t)
    print(f"[Fusion] Epoch {epoch+1}/{EPOCHS} — train_loss: {train_loss:.4f}")

[Fusion] Epoch 1/15 — train_loss: 0.0073
[Fusion] Epoch 2/15 — train_loss: 0.0037
[Fusion] Epoch 3/15 — train_loss: 0.0036
[Fusion] Epoch 4/15 — train_loss: 0.0035
[Fusion] Epoch 5/15 — train_loss: 0.0035
[Fusion] Epoch 6/15 — train_loss: 0.0034
[Fusion] Epoch 7/15 — train_loss: 0.0034
[Fusion] Epoch 8/15 — train_loss: 0.0034
[Fusion] Epoch 9/15 — train_loss: 0.0033
[Fusion] Epoch 10/15 — train_loss: 0.0033
[Fusion] Epoch 11/15 — train_loss: 0.0033
[Fusion] Epoch 12/15 — train_loss: 0.0033
[Fusion] Epoch 13/15 — train_loss: 0.0033
[Fusion] Epoch 14/15 — train_loss: 0.0033
[Fusion] Epoch 15/15 — train_loss: 0.0032


## Step 12 — Evaluate final fused classification

Full metric set per tamper type: confusion matrix (TP/TN/FP/FN), Accuracy, Precision,
Recall, F1, Specificity, False Positive Rate, False Negative Rate, ROC-AUC, PR-AUC, and
MCC.

**For this task specifically, Recall, Precision, F1, and FPR matter most** — a missed
theft (low recall) is lost revenue, but a false alarm (high FPR) sends someone on an
unnecessary field inspection, which has a real cost too. Accuracy on its own is close to
meaningless here since tamper events are rare — a model that never predicts a tamper at
all would still score high accuracy while being useless.

In [25]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
)

fusion_model.eval()
with torch.no_grad():
    test_logits = fusion_model(torch.tensor(X_fusion_test, dtype=torch.float32).to(device))
    test_proba = torch.sigmoid(test_logits).cpu().numpy()

fusion_predictions = (test_proba >= 0.5).astype(int)

metrics_rows = []

for i, label in enumerate(TARGET_COLS):
    y_true = Y_fusion_test[:, i]
    y_pred = fusion_predictions[:, i]
    y_proba = test_proba[:, i]

    # Confusion matrix — force both classes present so shape is always 2x2,
    # even if a rare tamper type has zero positives in this test split
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)          # a.k.a. sensitivity / TPR
    f1        = f1_score(y_true, y_pred, zero_division=0)

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0             # TNR
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0                     # = 1 - specificity
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0                     # = 1 - recall

    # ROC-AUC / PR-AUC / MCC are undefined if the test split has only one class present —
    # guard rather than let sklearn raise or silently mislead
    if len(np.unique(y_true)) < 2:
        roc_auc = np.nan
        pr_auc = np.nan
    else:
        roc_auc = roc_auc_score(y_true, y_proba)
        pr_auc = average_precision_score(y_true, y_proba)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan

    print(f"\n--- {label} ---")
    print(f"Confusion matrix (rows=true, cols=pred) [[TN FP] [FN TP]]:")
    print(cm)
    print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")
    print(f"Accuracy: {accuracy:.4f}  Precision: {precision:.4f}  Recall: {recall:.4f}  F1: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}  FPR: {fpr:.4f}  FNR: {fnr:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}  PR-AUC: {pr_auc:.4f}  MCC: {mcc:.4f}")

    metrics_rows.append({
        "Tamper Type": label, "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1": f1,
        "Specificity": specificity, "FPR": fpr, "FNR": fnr,
        "ROC-AUC": roc_auc, "PR-AUC": pr_auc, "MCC": mcc,
    })

metrics_df = pd.DataFrame(metrics_rows).set_index("Tamper Type")
metrics_df


--- Current Imbalance ---
Confusion matrix (rows=true, cols=pred) [[TN FP] [FN TP]]:
[[230133   2553]
 [  2493 106110]]
TP=106110  TN=230133  FP=2553  FN=2493
Accuracy: 0.9852  Precision: 0.9765  Recall: 0.9770  F1: 0.9768
Specificity: 0.9890  FPR: 0.0110  FNR: 0.0230
ROC-AUC: 0.9985  PR-AUC: 0.9977  MCC: 0.9659

--- Voltage Unbalance ---
Confusion matrix (rows=true, cols=pred) [[TN FP] [FN TP]]:
[[341216     17]
 [    50      6]]
TP=6  TN=341216  FP=17  FN=50
Accuracy: 0.9998  Precision: 0.2609  Recall: 0.1071  F1: 0.1519
Specificity: 1.0000  FPR: 0.0000  FNR: 0.8929
ROC-AUC: 0.9981  PR-AUC: 0.1725  MCC: 0.1671

--- Missing Potential ---
Confusion matrix (rows=true, cols=pred) [[TN FP] [FN TP]]:
[[341249      1]
 [    39      0]]
TP=0  TN=341249  FP=1  FN=39
Accuracy: 0.9999  Precision: 0.0000  Recall: 0.0000  F1: 0.0000
Specificity: 1.0000  FPR: 0.0000  FNR: 1.0000
ROC-AUC: 0.8711  PR-AUC: 0.0481  MCC: -0.0000

--- High Voltage ---
Confusion matrix (rows=true, cols=pred) [[TN FP] [F

,TP,TN,FP,FN,Accuracy,Precision,Recall,F1,Specificity,FPR,FNR,ROC-AUC,PR-AUC,MCC
Tamper Type,,,,,,,,,,,,,,
Current Imbalance,106110,230133,2553,2493,0.985215,0.976505,0.977045,0.976775,0.989028,0.010972,0.022955,0.998498,0.997676,0.965931
Voltage Unbalance,6,341216,17,50,0.999804,0.260870,0.107143,0.151899,0.999950,0.000050,0.892857,0.998056,0.172466,0.167098
Missing Potential,0,341249,1,39,0.999883,0.000000,0.000000,0.000000,0.999997,0.000003,1.000000,0.871149,0.048148,-0.000018
High Voltage,5,341221,23,40,0.999815,0.178571,0.111111,0.136986,0.999933,0.000067,0.888889,0.965429,0.061966,0.140770
Low Voltage,283,340911,21,74,0.999722,0.930921,0.792717,0.856278,0.999938,0.000062,0.207283,0.998520,0.921424,0.858911
Over Current,2,341225,0,62,0.999818,1.000000,0.031250,0.060606,1.000000,0.000000,0.968750,0.808540,0.060152,0.176761
Very Low PF,94973,245122,603,591,0.996501,0.993691,0.993816,0.993753,0.997546,0.002454,0.006184,0.999939,0.999844,0.991324
Neutral Disturbance,74,341208,2,5,0.999979,0.973684,0.936709,0.954839,0.999994,0.000006,0.063291,0.997931,0.951452,0.955007
CT Reversal,104516,236170,359,244,0.998233,0.996577,0.997671,0.997124,0.998482,0.001518,0.002329,0.999975,0.999957,0.995849


## Step 13 — Save everything
`smart_meter_labeled.csv` itself is never touched. All three stages saved separately.

In [ ]:
import joblib

torch.save(resnet_stage1.extractor.state_dict(), "model/resnet1d_extractor.pt")
torch.save(lstm_classifier.state_dict(), "model/lstm_classifier.pt")
torch.save(gnn_stage1.gnn.state_dict(), "model/feeder_gnn.pt")
torch.save(fusion_model.state_dict(), "model/fusion_classifier.pt")
joblib.dump(seq_scaler, "model/seq_scaler.joblib")
joblib.dump(tab_scaler, "model/tab_scaler.joblib")

metrics_df.to_csv("Dataset/fusion_evaluation_metrics.csv", index=False)

# Persist the exact config used for this run, so a separate evaluation script can
# reconstruct the identical test set later — recomputing GLOBAL_CUTOFF_TIME from the
# data again would only match if the CSV never changes, so it's saved explicitly instead.
import json
run_config = {
    "WINDOW_SIZE": WINDOW_SIZE,
    "RESNET_FILTERS": RESNET_FILTERS,
    "LSTM_UNITS": LSTM_UNITS,
    "MIN_METERS_FOR_GNN": MIN_METERS_FOR_GNN,
    "GNN_HIDDEN_DIM": GNN_HIDDEN_DIM,
    "GLOBAL_CUTOFF_TIME": str(GLOBAL_CUTOFF_TIME),
    "RAW_FEATURE_COLS": RAW_FEATURE_COLS,
    "TARGET_COLS": TARGET_COLS,
    "TIME_COL": TIME_COL,
    "ID_COL": ID_COL,
}
with open("model/run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

print("Saved resnet1d_extractor.pt, lstm_classifier.pt, feeder_gnn.pt, fusion_classifier.pt, scalers, fusion_evaluation_metrics.csv, run_config.json")
print("smart_meter_labeled.csv was not modified.")

Saved resnet1d_extractor.pt, lstm_classifier.pt, feeder_gnn.pt, fusion_classifier.pt, scalers, fusion_evaluation_metrics.csv, run_config.json
smart_meter_labeled.csv was not modified.


: 